In [3]:
%pip install pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 143.6 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [1]:
import random
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# 1. VERIFY RUNPOD GPU IS ACTIVE (Take a screenshot of this output!)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"=== RunPod Training Device: {device} ===")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# 2. DEFINE 2025 MALAYSIA MONTHLY CALENDAR & FESTIVALS
months_2025 = {
    1:  ("Jan 2025", "Chinese New Year Eve & Lou Sang", 1.45, 75, 4),
    2:  ("Feb 2025", "CNY Open Houses & Chap Goh Mei", 1.35, 70, 5),
    3:  ("Mar 2025", "Ramadan Buka Puasa Banquets", 0.95, 45, 6),
    4:  ("Apr 2025", "Hari Raya Aidilfitri Open Houses", 1.50, 80, 3),
    5:  ("May 2025", "Mid-Year Corporate Seminars", 0.90, 40, 7),
    6:  ("Jun 2025", "Gawai/Kaamatan & Trade Expos", 1.00, 42, 7),
    7:  ("Jul 2025", "Off-Peak Mid-Year Period", 0.80, 30, 8),
    8:  ("Aug 2025", "Merdeka Month Marathons & Carnivals", 1.30, 62, 6),
    9:  ("Sep 2025", "Malaysia Day & Tech Exhibitions", 1.25, 58, 6),
    10: ("Oct 2025", "Deepavali Festive Celebrations", 1.35, 65, 5),
    11: ("Nov 2025", "Year-End Conferences & Graduations", 1.30, 60, 6),
    12: ("Dec 2025", "Year-End Corporate Annual Dinners", 1.60, 90, 4),
}

# Event types: (Base Price/Pax, Cost/Pax, Chairs/100, Tables/100, Crew/100)
event_specs = {
    "Annual Dinner / Gala":    (120, 45, 105, 11, 4.0),
    "Festive Open House":      (95,  38, 90,  10, 3.5),
    "Trade Exhibition":        (85,  30, 75,  24, 3.0),
    "City Marathon / Run":     (65,  22, 25,  6,  2.2),
    "Grand Opening Ceremony":  (150, 55, 100, 12, 4.8),
    "Corporate Seminar":       (90,  32, 102, 10, 2.8),
}
event_names = list(event_specs.keys())

# 3. GENERATE 650+ HISTORICAL 2025 MALAYSIAN EVENT RECORDS
records = []
event_id = 1001

for month_num, (m_label, festival, season_mult, num_events, avg_ft_avail) in months_2025.items():
    for _ in range(num_events):
        ev_type = random.choice(event_names)
        ev_idx = event_names.index(ev_type)
        base_price, unit_cost, ch_ratio, tb_ratio, cr_ratio = event_specs[ev_type]

        # Guest count scales up during festive months
        base_pax = random.randint(100, 900)
        pax = int(round(base_pax * random.uniform(0.9, 1.15) * (1.15 if season_mult > 1.2 else 1.0), -1))

        # Calculate realistic historical equipment & workforce needs
        scale = pax / 100.0
        chairs = int(np.ceil(scale * ch_ratio * random.uniform(0.98, 1.05)))
        tables = int(np.ceil(scale * tb_ratio * random.uniform(0.98, 1.05)))
        total_crew = int(np.ceil(scale * cr_ratio * (1.15 if season_mult > 1.25 else 1.0)))

        # Full-time crew available after Orbyte eLeave (fewer available during major festivals)
        ft_available = max(2, min(8, int(round(random.gauss(avg_ft_avail, 1.0)))))
        part_timers = max(0, total_crew - ft_available)

        pt_daily_wage = 150 if season_mult >= 1.3 else 120
        revenue = pax * base_price
        total_cost = (pax * unit_cost) + (part_timers * pt_daily_wage)
        net_margin_pct = round(((revenue - total_cost) / revenue) * 100, 1)

        records.append({
            "Event_ID": f"EV2025-{event_id}",
            "Month_Num": month_num,
            "Month_Name": m_label,
            "Malaysia_Season": festival,
            "Seasonal_Multiplier": season_mult,
            "Event_Type": ev_type,
            "Event_Type_Code": ev_idx,
            "Guest_Pax": pax,
            "Chairs_Used": chairs,
            "Tables_Used": tables,
            "Total_Crew_Needed": total_crew,
            "FullTime_Available_Orbyte_eLeave": ft_available,
            "Part_Timers_Hired": part_timers,
            "Revenue_RM": revenue,
            "Total_Cost_RM": total_cost,
            "Net_Margin_Pct": net_margin_pct
        })
        event_id += 1

df = pd.DataFrame(records)
df.to_csv("malaysia_events_2025.csv", index=False)
print(f"Generated malaysia_events_2025.csv with {len(df)} records!")

# 4. TRAIN PYTORCH NEURAL NETWORK ON RUNPOD GPU
# Inputs: [Month_Num, Seasonal_Multiplier, Event_Type_Code, Guest_Pax, FullTime_Available_Orbyte_eLeave]
# Outputs: [Chairs_Used, Tables_Used, Total_Crew_Needed, Part_Timers_Hired]
X_cols = ["Month_Num", "Seasonal_Multiplier", "Event_Type_Code", "Guest_Pax", "FullTime_Available_Orbyte_eLeave"]
y_cols = ["Chairs_Used", "Tables_Used", "Total_Crew_Needed", "Part_Timers_Hired"]

X_raw = df[X_cols].values.astype(np.float32)
y_raw = df[y_cols].values.astype(np.float32)

# Normalize features for stable neural network training
X_mean, X_std = X_raw.mean(axis=0), X_raw.std(axis=0) + 1e-6
X_norm = (X_raw - X_mean) / X_std

X_tensor = torch.tensor(X_norm, device=device)
y_tensor = torch.tensor(y_raw, device=device)

model = nn.Sequential(
    nn.Linear(5, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 4)
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
loss_fn = nn.MSELoss()

for epoch in range(1, 501):
    optimizer.zero_grad()
    preds = model(X_tensor)
    loss = loss_fn(preds, y_tensor)
    loss.backward()
    optimizer.step()
    if epoch % 100 == 0:
        print(f"Epoch {epoch}/500 | GPU MSE Loss: {loss.item():.2f}")

# Save trained model weights + normalization stats for Replit
torch.save({
    "model_state": model.cpu().state_dict(),
    "X_mean": X_mean.tolist(),
    "X_std": X_std.tolist(),
    "event_names": event_names
}, "event_model_2025.pt")
print("Saved trained model to event_model_2025.pt!")

=== RunPod Training Device: cuda ===
GPU Name: NVIDIA RTX A5000
Generated malaysia_events_2025.csv with 717 records!
Epoch 100/500 | GPU MSE Loss: 5696.16
Epoch 200/500 | GPU MSE Loss: 4603.47
Epoch 300/500 | GPU MSE Loss: 4032.80
Epoch 400/500 | GPU MSE Loss: 3633.46
Epoch 500/500 | GPU MSE Loss: 3420.12
Saved trained model to event_model_2025.pt!
